# EarlyGate vs gold (`casos_gold_criticidad_v2`)

Evalúa **`EarlyGate.decide`** frente a **`gold_mode`** con métricas (accuracy, F1, κ), tablas y gráficas.

**Por qué existe también el `.py`:** el script `scripts/eval_early_gate_corpus.py` sirve para ejecutar en terminal, CRON o sin interfaz; este notebook es para **ver la corrida**, inspeccionar filas y presentar resultados.

**Requisitos:** kernel con `scikit-learn`, `requests`, `python-dotenv`, `tqdm`, `pandas`, `matplotlib`, `seaborn`. Activar el `.venv` del repo (`pip install -r requirements.txt`).

**Cwd:** debe poder encontrarse `agentes/core/early_gate.py` subiendo desde el directorio de trabajo actual (por ejemplo, abrir la raíz del repositorio en Cursor).

In [ ]:
%matplotlib inline

import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "agentes" / "core" / "early_gate.py").is_file():
            return p
    raise FileNotFoundError(
        "No se encontró la raíz del repo (donde está agentes/core/early_gate.py)."
    )


REPO_ROOT = find_repo_root()
AGENTES_DIR = REPO_ROOT / "agentes"
CORPUS = REPO_ROOT / "data" / "criticidad_cases" / "casos_gold_criticidad_v2.jsonl"
REPORTS_DIR = REPO_ROOT / "reports"

load_dotenv(REPO_ROOT / ".env")
sys.path.insert(0, str(AGENTES_DIR))

if not os.getenv("DEEPSEEK_API_KEY", "").strip():
    raise RuntimeError("Defina DEEPSEEK_API_KEY en .env en la raíz del repo (o en el entorno).")

print("REPO_ROOT =", REPO_ROOT)
print("Corpus:", CORPUS, "→ existe:", CORPUS.is_file())

In [ ]:
# --- Parámetros de la corrida ---
LIMIT = None  # p.ej. 10 para prueba rápida; None = las 300 filas
DELAY_SEC = float(os.getenv("EARLY_EVAL_DELAY_SEC", "0.35"))
SAVE_TO_REPORTS = True  # guarda JSON/CSV/PNG en reports/ como el script


def load_jsonl(path: Path, limit: int | None) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
            if limit is not None and len(rows) >= limit:
                break
    return rows


rows = load_jsonl(CORPUS, LIMIT)
len(rows)

In [ ]:
from tqdm.auto import tqdm

from core.early_gate import EarlyGate

labels = ["play", "pausa", "stop"]
gate = EarlyGate()
y_true: list[str] = []
y_pred: list[str] = []
details: list[dict] = []

t0 = time.perf_counter()
for i, row in enumerate(tqdm(rows, desc="EarlyGate.decide")):
    req = row.get("requirement", "")
    gold = row.get("gold_mode", "")
    rid = row.get("id", str(i))
    out = gate.decide(req)
    pred = out.get("gold_mode", "pausa")
    if pred not in labels:
        pred = "pausa"
    y_true.append(gold)
    y_pred.append(pred)
    details.append(
        {
            "id": rid,
            "gold_mode": gold,
            "predicted_mode": pred,
            "match": gold == pred,
            "source": out.get("source"),
            "confidence": out.get("confidence"),
            "doubt": out.get("doubt"),
        }
    )
    if i + 1 < len(rows) and DELAY_SEC > 0:
        time.sleep(DELAY_SEC)

elapsed = time.perf_counter() - t0
print(f"Tiempo total: {elapsed:.1f} s ({len(rows)} casos)")

In [ ]:
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
f1_per = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
kappa = cohen_kappa_score(y_true, y_pred, labels=labels)
cm = confusion_matrix(y_true, y_pred, labels=labels)

prec, rec, f1_prf, support_vec = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, zero_division=0
)

display(
    pd.DataFrame(
        {
            "métrica": ["accuracy", "f1_macro", "cohen_kappa"],
            "valor": [acc, f1_macro, kappa],
        }
    )
)

df_class = pd.DataFrame(
    {
        "clase": labels,
        "precision": prec,
        "recall": rec,
        "f1": f1_prf,
        "support": support_vec,
    }
)
display(df_class)

df_src = pd.DataFrame(details)
by_src = (
    df_src.groupby("source", dropna=False)
    .agg(n=("id", "count"), correct=("match", "sum"))
    .reset_index()
)
by_src["accuracy"] = by_src["correct"] / by_src["n"]
display(by_src.sort_values("n", ascending=False))

print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
    ax=ax,
)
ax.set_xlabel("Predicho (EarlyGate)")
ax.set_ylabel("Gold")
ax.set_title("Matriz de confusión")
plt.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(figsize=(7, 4))
x = np.arange(len(labels))
ax2.bar(x, [f1_per[i] for i in range(3)], width=0.5, color="steelblue")
ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.set_ylim(0, 1.05)
ax2.set_ylabel("F1")
ax2.set_title("F1 por clase")
plt.tight_layout()

if SAVE_TO_REPORTS:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    out_dir = REPORTS_DIR / f"early_gate_eval_nb_{stamp}"
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / "confusion_matrix.png", dpi=150, bbox_inches="tight")
    fig2.savefig(out_dir / "f1_per_class.png", dpi=150, bbox_inches="tight")
    df_src.to_csv(out_dir / "predictions.csv", index=False)
    summary = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "n_samples": len(rows),
        "elapsed_sec": round(elapsed, 2),
        "metrics": {
            "accuracy": float(acc),
            "f1_macro": float(f1_macro),
            "cohen_kappa": float(kappa),
        },
    }
    (out_dir / "summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("Archivos guardados en:", out_dir)

plt.show()

In [ ]:
# Vista rápida de errores (primeras filas)
df_err = df_src[~df_src["match"]][
    ["id", "gold_mode", "predicted_mode", "source", "confidence"]
].head(20)
display(df_err)